# New_Apple_Import Py File

## Uploading New Apple Raw Data

In [ ]:
import os
import datetime as dt
import sys
import pandas as pd
from sqlalchemy import create_engine, text
import time
import numpy as np
from zoneinfo import ZoneInfo
from datetime import datetime

engine = create_engine(f"sqlite:///habits.db")

# def max_date():

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    max_date_db = pd.read_sql_query(
        text("""SELECT MAX(DATETIME(adr.startDate)) as 'Max_Raw_UTC_Start_Date'
                FROM apple_data_raw adr"""), 
        connection,
        parse_dates=['Max_Raw_UTC_Start_Date'])   


max_date_db["Max_Raw_UTC_Start_Date"] = (
    pd.to_datetime(max_date_db["Max_Raw_UTC_Start_Date"])
    .dt.tz_localize("UTC"))

max_UTC_date_db = max_date_db["Max_Raw_UTC_Start_Date"].iloc[0]

In [ ]:
# def upload_new_raw_apple_data(inital_max_date):

import pandas as pd
import os 
# ETL step of gathering new data from apple health and upploading it almost as it is apple_data_raw

# Defining the subddirectroy where apple docs live
directory = "apple/"

# Initializing variable
apple_file = "apple_health_export_2025-12-14.csv"

# Read in new CSV File, low_memory=False to avoid dtype warnings
apple = pd.read_csv(os.path.join(directory, apple_file), low_memory=False)

# Fitler the source to only my Apple Watch (Could pull in other devices later like iPhone or gamin)
apple = apple[(apple['sourceName'] == "Ryan’s Apple\xa0Watch") | (apple['sourceName'].isna())]

# Strip the "HKWorkoutActivityType" prefix from the workout types
apple["workoutActivityType"] = apple["workoutActivityType"].str.replace("HKWorkoutActivityType", "")

# Drop the UUID column, apple added this since last 7/18 upload
apple.drop(columns=['uuid'], inplace=True)

# Creating a copy of apple dataframe to avoid settingwithcopy warnings when converting datetime
apple_copied = apple.copy()

# Cleaning step helpful for troubleshooting
del apple

# Convert date columns to datetime format including UTC timezone
datetime_cols = ['startDate', 'endDate', 'creationDate']
for col in datetime_cols:
    apple_copied[col] = pd.to_datetime(apple_copied[col], format='%Y-%m-%d %H:%M:%S %z', utc=True)

# Using the exisiting max date in my db to filter to only new data from apple 
apple_copied = apple_copied[apple_copied["startDate"] > max_UTC_date_db] # dt_utc_test  max_UTC_date_db

In [ ]:
# Appending the new raw data to the apple_data_raw table
apple_copied.to_sql(name="apple_data_raw", con=engine, if_exists='append', index=False)

### Testing/Troubleshooting code

In [ ]:
# Little sanity checking step for troubleshooting
min(apple_copied["startDate"])

In [ ]:
# Filler UTC Datetime variable for testing purposes
from datetime import datetime, timezone

dt_utc_test = datetime(2025, 12, 10, tzinfo=timezone.utc)
max_UTC_date_db = dt_utc_test

## Converting Apple Raw Data to Apple workouts data 

In [ ]:
import os
import datetime as dt
import sys
import pandas as pd
from sqlalchemy import create_engine, text
import time
import numpy as np
from zoneinfo import ZoneInfo
from datetime import datetime

engine = create_engine(f"sqlite:///habits.db")

#def Read_Apple_Workouts(max_UTC_date_db):

# Reading in only the newly updated data from apple_data_raw (using inital max_UTC_date_db prior to uploading new raw data)
with engine.connect() as connection: 
    apple_raw = pd.read_sql_query( 
        text("""SELECT workoutActivityType, startDate, type, maximum, minimum, sum, average, duration, durationUnit 
        FROM apple_data_raw
        WHERE startDate > :max_date"""), # Kinda odd looks like max_UTC_date_db gets carried over into apple_raw (shouldn't matter though
        connection,                     # because we're only fitering to actual workouts)
        params={"max_date": str(max_UTC_date_db)}, # Using the str version of the datetime for filtering
        parse_dates=['startDate']) # Parse date here takes out need for pd.to_datetime below 

apple_raw["startDate"] = apple_raw["startDate"].dt.tz_localize("UTC")

# Filters df to include only rows where at least one of the workout-related columns is not missing (NaN).
# .notna() Check for Non-Nulls in selected columns 
# .any(axis=1) aggregates rows where there's at least one non-missing value 
apple_workouts = apple_raw[apple_raw.notna().any(axis=1)] 

In [ ]:
#def aw_to_long(apple_workouts):

# Reshape the apple_workouts DataFrame from wide format (one row per workout with multiple measurement columns) to long format (one row per measurement per workout).
df_long = pd.melt(apple_workouts, 
                    id_vars=['startDate', 'workoutActivityType', 'type', 'duration', 'durationUnit'],
                    value_vars=['maximum', 'minimum', 'sum', 'average'], # turned into rows via where the name goes into 'measurement_type' and actual values to the 'value' column
                    var_name='measurement_type', 
                    value_name='value')

# Removing rows where either type or value are null (no meaningful data here)
df_long = df_long[(df_long['type'].notna()) | (df_long['value'].notna())]

# Handle duration rows separately and combine
duration_rows = apple_workouts[apple_workouts['duration'].notna() & apple_workouts['type'].isna()].copy()
duration_rows = duration_rows.assign(
    measurement_type='total',
    value=duration_rows['duration'],
    type='Duration')[['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']]

# Combine and clean
aw_long = pd.concat([df_long[df_long['value'].notna()][['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']],
    duration_rows], ignore_index=True)

# Rename columns for clarity
aw_long.rename(columns={'startDate': 'StartDate', 'workoutActivityType': 'activity', 'type': 'metric', # Why do I rename StartDate to be different than apple data raw?
                       'measurement_type': 'measurement_type', 'value': 'value', 'durationUnit': 'd_unit'}, inplace=True)

# cleaning value column to only go out 2 decimal places
aw_long['value'] = aw_long['value'].round(2)

aw_long.sort_values('StartDate', ascending=False, inplace=True)

In [ ]:
def max_workout_id():
        #Gathering the max workout_id from existing apple_workouts table
        with engine.connect() as connection:
                max_workout_id_df = pd.read_sql_query(
                        text("""SELECT max(workout_id) as workout_id 
                                FROM apple_workouts
                                WHERE activity IS NOT NULL"""), # Need to filter on activity because otherwise workout_id is NA
                        connection)  

                # Grabbing singlular max workout_id from dataset to filter new data off of  
                max_workout_id = int(max_workout_id_df['workout_id'][0]) # Maybe make this an INT variable?

        return max_workout_id

#max_workout_id = max_workout_id()

In [ ]:
#def add_workout_id(aw_long):

# We're able to easily get unique workouts by date because aw_long only has activity filled for the duration metric which every workout has 1 of 
activity_map = aw_long[aw_long['activity'].notnull()][["StartDate", "activity"]]

# Sorting values by date and resetting index (easy way to grab a unique id so long as I'm sorting correctly)
activity_map = activity_map.sort_values('StartDate').reset_index(drop=True)

# Creating a new variable based on the index  
# Previous null values probably from Garmin are keeping workout_id from being an integer in aw_final
activity_map['workout_id'] = activity_map.index.astype(int) + max_workout_id()  # Start IDs from 1

# Merge activity back onto the main dataframe using startDate
aw_final = pd.merge(aw_long, activity_map, on='StartDate', how='left', suffixes=('', '_Specifier'))

# Drop the original activity column
aw_final.drop(columns=['activity'], inplace=True)

# Rename the helper column to activity since that's actually what we'll want
aw_final.rename(columns={'activity_Specifier': 'activity'}, inplace=True)

# Adding new activity type variable for minutes calculation 
aw_final['activity_type'] = np.where(aw_final['activity'].isin(['Running', 'Cycling', 'Swimming', 'Walking']), 'Cardio',
    np.where(aw_final['activity'] == 'TraditionalStrengthTraining', 'Weights', None))

In [ ]:
#def final_upload(aw_final):
aw_final.to_sql(name="apple_workouts", con=engine, if_exists='append', index=False)

## Adding .fit files to specific bike tables in habit db

#### Reading in fit data for testing table insertion functions

In [ ]:
import os
from fitparse import FitFile
import pandas as pd
from sqlalchemy import create_engine, text 

engine = create_engine(f"sqlite:///habits.db")

bike_file = r"bike_files\NWzvfhQlS6dpEJ2T5zUbg3LJBoXv51npbtW43hzr.fit"

fitfile = FitFile(bike_file)

# Initializing rows
rows = []

# Iterate over all record messages
for record in fitfile.get_messages('record'):
    row = {}

    # Each record contains multiple data fields
    for field in record:
        row[field.name] = field.value

    rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)

# Dropping heart rate because it's blank
raw_bike_data = df.drop(columns="heart_rate")

### bike_workout table

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import date

engine = create_engine(f"sqlite:///habits.db")
## Defining server upload time once to use in multiple places
today_upload_date = date.today().strftime("%m-%d-%Y")

### Bike Workout table
#def bike_workout_upload(raw_bike_data, today_upload_date):

# Converting timestamp to utc datetime format for sqlite db
raw_bike_data["timestamp"] = raw_bike_data["timestamp"].astype(str) + ".000000"

# Grabbing min and max times for the bike workout
# timestamp from .fit file is already in UTC
max_time = max(raw_bike_data["timestamp"])
min_time = min(raw_bike_data["timestamp"])

engine = create_engine(f"sqlite:///habits.db")
with engine.begin() as conn:
    # Bike Workout table
    result = conn.execute(text("""INSERT INTO bike_workout (start_time, end_time, upload_date)
                        VALUES (:start_time, :end_time, :upload_date)"""),
                        {"start_time": str(min_time), "end_time": str(max_time), "upload_date": str(today_upload_date) })
    
    workout_id = result.lastrowid  # Get the ID of the workout we just inserted

#    return workout_id

### bike_units table

Mappings is something I should go back and dynamcially fill

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text 

engine = create_engine(f"sqlite:///habits.db")

#def bike_units_upload(raw_bike_data):

# Data cleaning step to conver the data to long format

## Statically Defining the metric-to-unit mappings
metric_unit_map = {
    'altitude': 'meters',
    'cadence': 'revolutions_per_minute',
    'distance': 'meters',
    'enhanced_altitude': 'meters',
    'enhanced_speed': 'meters_per_second',
    'position_lat': 'degrees',
    'position_long': 'degrees',
    'power': 'watts',
    'speed': 'meters_per_second'
}

## Statically defining the unit to abbreviation mappings
unit_abv_map = {
    'meters': 'm',
    'revolutions_per_minute': 'rpm',
    'meters_per_second': 'm/s',
    'degrees': 'deg',
    'watts': 'W',
}

## Melt the dataframe
bike_data_long = pd.melt(raw_bike_data, 
            id_vars=['timestamp'], 
            value_vars=['altitude', 'cadence', 'distance', 'enhanced_altitude',
                        'enhanced_speed', 'position_lat', 'position_long',
                        'power', 'speed'], 
            var_name="metric", 
            value_name="value")


## Add unit columns using map
bike_data_long['unit_name'] = bike_data_long['metric'].map(metric_unit_map) 
bike_data_long['metric_unit'] = bike_data_long['unit_name'].map(unit_abv_map)


# Uploading data to the bike units df only if it's new 

## Filtering bike_data_long to only revlavent columns associated with bike_units table
bike_units_df = bike_data_long[["metric", "unit_name", "metric_unit"]].drop_duplicates() # drop_dups to get unique values

engine = create_engine(f"sqlite:///habits.db")

for _, row in bike_units_df.iterrows():
    metric_type = row["metric"]
    metric_unit = row["metric_unit"]
    unit_name = row["unit_name"]

    with engine.begin() as conn:
        ## Insert only if it doesn't already exist
        conn.execute(text("""
            INSERT OR IGNORE INTO bike_units (metric_type, metric_unit, unit_name)
            VALUES (:metric_type, :metric_unit, :unit_name)"""), 
            {"metric_type": metric_type, "metric_unit": metric_unit, "unit_name": unit_name})
            
#    return bike_data_long
            

### bike_metrics table

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text 

engine = create_engine(f"sqlite:///habits.db")

#def bike_metrics_upload(bike_data_long, workout_id):

# Selecting data only relavent to bike_metrics table insertion 
bike_metrics_df = bike_data_long[["timestamp", "metric", "value"]].drop_duplicates().copy()

engine = create_engine(f"sqlite:///habits.db")

# workout_id comes from above insertion code
bike_workout_id = workout_id

# Grabbing all the metric types from the metrics table which I'll use to filter incoming bike data and insert into bike_metrics
with engine.begin() as conn:
   bike_units = pd.read_sql_query(text("""SELECT id, metric_type 
                                       FROM bike_units"""), conn)

bike_metrics_df["workout_id"] = bike_workout_id

# Looping through each bike metric and uploading data from new bike file with the appropriate bike metric id 
for _, row in bike_units.iterrows():
   metric_name = row["metric_type"]
   
   bike_metrics_upload_df = bike_metrics_df[bike_metrics_df["metric"] == metric_name].copy()

   # Now that we're filtered by metric name we can just insert the id for that metric
   bike_metrics_upload_df["metric_id"] = row["id"]
   
   # After filtering by metric column droppping it because now we have it's ID!
   bike_metrics_upload_df.drop(columns=["metric"], inplace=True)

   # Appending the new raw data to the apple_data_raw table
   bike_metrics_upload_df.to_sql(name="bike_metrics", con=engine, if_exists='append', index=False)

### Reading in new fit files and applying uploads to each table

In [ ]:
# Processing all new fit files (that don't have date already attached) and inserting appropriate data into each table (functionized)
import os
from fitparse import FitFile
import pandas as pd
from sqlalchemy import create_engine, text 

# Directory where all fit files are saved
directory = r"bike_files"
engine = create_engine(f"sqlite:///habits.db")

#def upload_new_fit_files(directory, today_upload_date):

# Initializing file number counter to start at the first file
file_number = 0
total_unprocessed_files = len([file for file in os.listdir(directory) if "Bike Workout" not in file])

for file_path in os.listdir(directory):
    # Filtering to fit files that I have not processed yet (All processed files get renamed)
    if "Bike Workout" not in file_path:
        bike_fit_file = os.path.join(directory, file_path)

        # This is where python creates the connection to the fit file and reads it in
        fitfile = FitFile(bike_fit_file)

        # Initializing rows
        rows = []

        # Iterate over all record messages
        for record in fitfile.get_messages('record'):
            row = {}

            # Each record contains multiple data fields
            for field in record:
                row[field.name] = field.value

            rows.append(row)

        # Create DataFrame
        df = pd.DataFrame(rows)

        # Dropping heart rate because it's blank
        raw_bike_data = df.drop(columns="heart_rate")

        # Putting bike data into appropriate tables
        ## Inserting data into the Bike workouts table
        workout_id = bike_workout_upload(raw_bike_data, today_upload_date)

        ## Inserting data into the Bike units table
        bike_data_long = bike_units_upload(raw_bike_data)    

        ## Inserting data into the Bike metrics table
        bike_metrics_upload(bike_data_long, workout_id)

        # For each fit file changing the name so I don't read it in again  
        for record in fitfile.get_messages('record'):
            for field in record:
                if field.name == "timestamp":     #, field.value, field.units
                    file_date = field.value
                    # String formatting the date variable
                    file_date = file_date.strftime('%m-%d-%Y')
                    break
            if file_date:
                break

        # Important to discard the connection to fitfile before saving over it otherwise os.rename will error out because python is still using the file
        del fitfile

        # This would be bad if we can't find timestamp data
        if not file_date:
            print(f"No timestamp found for {file_path}")
            continue
        
        # Renaming current file to fit naming schema: 01-14-2026 Bike Workout
        os.rename(bike_fit_file, os.path.join(directory, f"{file_date} Bike Workout.fit"))

        file_number += 1
        
        # Readable step to ensure correct number of files are processed
        print(f"Processed file '{file_path}', number {file_number} out of {total_unprocessed_files}")

## Using newly inserted bike data to appened apple_workouts for cycling workouts

In [ ]:
import pandas as pd
from sqlalchemy import text, create_engine

engine = create_engine(f"sqlite:///habits.db")

# Grabbing dates of all bike workouts from apple
#def get_apple_cycle_data():
with engine.begin() as conn:
    apple_cycle = pd.read_sql_query(
            text("""SELECT StartDate, activity, workout_id 
                    FROM apple_workouts
                    where activity = 'Cycling' and metric = 'Duration'
                    order by StartDate desc"""), conn,
            parse_dates=["StartDate"],
            dtype={"workout_id": "Int64"})

    # Localizing to UTC timezone for merging later
    apple_cycle["StartDate"] = apple_cycle["StartDate"].dt.tz_localize("UTC")

    # Abstracting to date for merge to .fit data (won't merge perfectly on datetime)
    apple_cycle["apple_date"] =  apple_cycle["StartDate"].dt.date

#        return apple_cycle

In [ ]:
# Grabbing total distance and average wattage from .fit files bike workouts we just uploaded and combining
import pandas as pd
from sqlalchemy import text, create_engine

engine = create_engine(f"sqlite:///habits.db")

# Grabbing total distance and average wattage from .fit files bike workouts we just uploaded
#def get_fit_cycle_data(today_upload_date):
with engine.begin() as conn:
    fit_distance = pd.read_sql_query(
        text("""SELECT bw.id, bw.start_time, MAX(bm.value) as 'max_distance'
                from bike_workout bw 
                LEFT OUTER JOIN bike_metrics bm 
                on bm.workout_id = bw.id 
                LEFT OUTER JOIN bike_units bu 
                on bu.id = bm.metric_id
                where bu.metric_type = 'distance' AND bw.upload_date = :today_upload_date 
                GROUP BY bw.id"""), conn,
        params={"today_upload_date": str(today_upload_date)},
        parse_dates=["start_time"])
    
    fit_watts = pd.read_sql_query(
        text("""SELECT bw.id, ROUND(AVG(bm.value), 2) as 'avg_watts'
                from bike_workout bw 
                LEFT OUTER JOIN bike_metrics bm 
                on bm.workout_id = bw.id 
                LEFT OUTER JOIN bike_units bu 
                on bu.id = bm.metric_id
                where bu.metric_type = 'power' AND bw.upload_date = :today_upload_date
                GROUP BY bw.id"""), conn,
        params={"today_upload_date": str(today_upload_date)},
        parse_dates=["start_time"])

# Localizing to UTC timezone for merging because this is the start time var that will stay
fit_distance["start_time"] = fit_distance["start_time"].dt.tz_localize("UTC")

fit_distance["fit_date"] = fit_distance["start_time"].dt.date

# Merging various calculations of bike metric data together at bike workout level into singular df for upload
fit_total = pd.merge(fit_distance, fit_watts, on="id", how="left")

#    return fit_total

For this below code to work there needs to be associated indoor cycling workouts in the apple workouts df already 

In [ ]:
import pandas as pd
from sqlalchemy import text, create_engine

# Lopping through each variable option to upload those sepcific options to apple workouts table
#def upload_fit_to_apple(fit_total, apple_cycle):

# Using fit file data to find corresponding apple bike workouts 
combined = pd.merge(fit_total, apple_cycle, left_on="fit_date", right_on="apple_date", how="left")

# Meters to miles conversion
combined["max_distance_miles"] = (combined["max_distance"] / 1609).round(2)

# Converting start date to sqlite appropriate format
combined["StartDate"] = combined["StartDate"].dt.strftime('%Y-%m-%d %H:%M:%S') + ".000000"

# Dropping unnecessary/helper columns
combined.drop(columns=["start_time", "max_distance", "fit_date", "apple_date"], inplace=True)

## Defining dict with variable options   
variables = {"max_distance_miles": {"metric":"DistanceCycling", "measurement_type": "sum"}, 
            "avg_watts": {"metric":"AvgWatts", "measurement_type": "average"}}

with engine.begin() as conn:
    for var_name, var_config in variables.items() :
        temp_df = combined[combined[var_name].notnull()][["StartDate",  "activity", "workout_id", var_name]]

        temp_df["metric"] = var_config["metric"]
        temp_df["measurement_type"] = var_config["measurement_type"]
        temp_df["activity_type"] = "Cardio"

        for _, row in temp_df.iterrows():

            conn.execute(text("""
                    INSERT INTO apple_workouts (StartDate, activity, metric, measurement_type, value, d_unit, activity_type, workout_id)
                    VALUES (:StartDate, :activity, :metric, :measurement_type, :value, :d_unit, :activity_type, :workout_id)"""), {
                    'StartDate': str(row["StartDate"]),
                    'activity': row["activity"],
                    'metric': row["metric"],
                    'measurement_type': row["measurement_type"],
                    'value': row[var_name],  # Taking the variable name from the dict key
                    'd_unit': None,
                    'activity_type': row["activity_type"],
                    'workout_id': row["workout_id"]
                })


# Data Cleaning + Visualization PY

## Reading in Apple Workouts db from SQLite

In [ ]:
import os
import datetime as dt
import sys
import pandas as pd
from sqlalchemy import create_engine, text
import time
import numpy as np
from zoneinfo import ZoneInfo
from datetime import datetime
import plotly.express as px

# Read in cleaned Apple Workouts data 
#def Read_Apple_Workouts():

engine = create_engine(f"sqlite:///habits.db")

# Read in the data and parse datetime columns
with engine.connect() as connection:
    aw_all = pd.read_sql_query(
        text("""SELECT * FROM apple_workouts"""),
        connection)
        #dtype={"workout_id": "Int64"}) # Converting workout_id to Int64 upon entry, will need again if workout_id includes blanks
    
# Converting the string datetime variable from db to with timezone normalization to UTC then converting to EST
# This is needed because of combination of EST and EDT in the data 
# I could do this with a parse dates function in pd.read_sql_query() function above
date_cols = ["StartDate"]
for col in date_cols:
    aw_all[col] = (
        pd.to_datetime(aw_all[col])
        .dt.tz_localize("UTC")
        .dt.tz_convert("US/Eastern"))

### Adding in categorical variables for month and week

In [ ]:
# Creating a month categorical variable
# Create a string label for display
aw_all['month'] =  aw_all['StartDate'].dt.strftime('%b %Y') # Need .dt for series/panda 

# Certainly 1 way to get month from datetime but keep it as a datetime variable lol
aw_all['month_date'] =  pd.to_datetime(aw_all['month'], format='%b %Y') # Need .dt for series/panda

# Need to localize datetime to US/Eastern
aw_all['month_date'] = aw_all['month_date'].dt.tz_localize("US/Eastern")

# Set 'month' as a categorical(factor variable) with order based on 'month_date'
month_order = aw_all.sort_values('month_date')['month'].unique()
aw_all['month'] = pd.Categorical(aw_all['month'], categories=month_order, ordered=True)


# Creating a week categorical variable
# calculating out necessary time metrics when reading in apple workouts data
## calculates how many days each weekday is from monday and subtracts that from the orignal date to get back to start of the week Monday
aw_all['week_date'] = (
    aw_all['StartDate'] - pd.to_timedelta(aw_all['StartDate'].dt.weekday, unit="D") # dt.weekday uses Monday as 0 
).dt.normalize() # Resulting date is in EST (I believe this is the correct move when filtering later on)

    # Create a string label for display
aw_all['week'] = aw_all['week_date'].dt.strftime('%b %d')

# Set 'week_label' as a categorical(factor variable) with order based on 'week_period'
week_order = aw_all.sort_values('week_date')['week'].unique()
aw_all['week'] = pd.Categorical(aw_all['week'], categories=week_order, ordered=True)

## Anlysis

### Filling missing combinations function

In [ ]:
from itertools import product
# Function that fills in missing data for each grouped by df 
def fill_missing_combinations(
    original_df,
    aggregated_df,
    time_col,
    category_col,
    value_cols,
    time_filter=None,
    category_values=None):

    # Step 1: Filter time periods if needed
    if time_filter:
        time_values = original_df[time_filter(original_df)].loc[:, time_col].unique()
    else:
        time_values = original_df[time_col].unique()
    # Step 2: Get all categories
    if category_values:
        category_values = category_values
    else:
        category_values = aggregated_df[category_col].unique()
    # Step 3: Create full index
    full_index = pd.DataFrame(
        product(category_values, time_values),
        columns=[category_col, time_col])
    # Step 4: Merge with aggregated data
    filled_df = pd.merge(full_index, aggregated_df, on=[category_col, time_col], how='left')
    # Step 5: Fill NaNs with 0 for specified value columns
    for col in value_cols:
        filled_df[col] = filled_df[col].fillna(0)

    return filled_df.sort_values(by=[time_col, category_col]).reset_index(drop=True)

### Establishing consistent time filters

In [ ]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from zoneinfo import ZoneInfo
from plotnine import ggplot, aes, geom_bar, geom_text, position_dodge, scale_fill_brewer, scale_y_continuous, labs, theme_seaborn, theme, scale_fill_manual, element_line, geom_hline, geom_line, \
geom_point, scale_x_datetime, element_text, element_blank, geom_boxplot, stat_summary, scale_color_manual, element_rect, guides, guide_legend

today = datetime.now(tz=ZoneInfo("US/Eastern"))

l_3_m = today - relativedelta(weeks=14)
l_7_m = today - relativedelta(months=7)
l_1_y = today - relativedelta(years=1)
l_2_w = today - relativedelta(weeks=2)

### DF and Visualizations pairs

#### Frequency bar chart grouped by exercise type and month

In [ ]:
# def gen_month_freq_df(aw_all, filter1):
activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

df_counts = (aw_all[(aw_all['metric'] == 'Duration') & (aw_all['activity'].isin(activities)) & 
                    (aw_all['month_date'] > l_7_m)]
                    .groupby(['month_date', 'activity'], observed=True) # Include only combinations that actually occur in the data
                    .size() 
                    .reset_index(name='n'))

month_count_data = fill_missing_combinations(
    original_df=df_counts,
    aggregated_df=df_counts,
    time_col='month_date',
    category_col='activity',
    value_cols=['n'])

month_count_data['n'] = month_count_data['n'].astype(int)  # Convert to int for better readability

# Adding a label for 'TraditionalStrengthTraining' top shorten it for the graph output
month_count_data['activity'] = month_count_data['activity'].replace({'TraditionalStrengthTraining': 'Weights'})

# Merging in categorical month label
month_count_data = pd.merge(month_count_data,
    aw_all[['month', 'month_date']].drop_duplicates(),
    on='month_date',
    how='left')

freq_cush = 2

session_y_max = int(month_count_data["n"].max() + freq_cush)

In [ ]:
############### Frequency bar chart grouped by exercise type and month ###############
#month_count_data = gen_month_freq_df(aw_final, l_7_m)

#def Monthly_Freq_BarChart():
    
plot = (ggplot(month_count_data, aes(x='month', y='n', fill='activity')) + 
    geom_bar(stat='identity', position='dodge', color = "Black") +
    
    geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 
    
    scale_fill_brewer(type='qual', palette='Set2') +
    scale_y_continuous(breaks = range(0, session_y_max + 1, 2),
                    limits = [0, session_y_max]) +

    labs(title='',
        x='',
        y='Sessions',
        fill='Activity',
        color='Goal') +

    theme(
        figure_size=(10, 5),
        legend_position= (.5, .95),
        legend_title=element_text(color="#E8EFF5", size=11, weight='bold'),
        legend_direction='horizontal',
        legend_text=element_text(color="#A0B4C8", size=9),
        legend_background= element_blank(), #element_rect(fill="#1E2A38", color=None),
        legend_key=element_blank(), # Removes boxing/shading around lines in legend
        #plot_background   = element_rect(fill="#141C26", color=None),
        #panel_background  = element_rect(fill="#1E2A38", color=None),
        #panel_grid_major  = element_line(color="#2E3D4F", size=0.5),
        panel_grid_minor_y = element_line(color = "White", linetype = "solid"),
        axis_text         = element_text(color="#6B8299", size=9),
        axis_title        = element_text(color="#A0B4C8", size=10) ) +

    
    theme(figure_size=(10, 5)))

plot

#### Frequency bar chart grouped by exercise type and week (Deprecated)

In [ ]:
############### Frequency bar chart grouped by exercise type and week ###############
# def gen_week_freq_df(aw_final, filter2):

activities = ["Walking", "Cycling", "TraditionalStrengthTraining", "Running", "Swimming"]

df_counts = (aw_all[(aw_all['metric'] == 'Duration') & (aw_all['activity'].isin(activities)) & (aw_all['week_date'] > l_3_m)]
    .groupby(['week_date', 'activity'])
    .size()
    .reset_index(name='n'))

week_count_data = fill_missing_combinations(
    original_df=df_counts,
    aggregated_df=df_counts,
    time_col='week_date',
    category_col='activity',
    value_cols=['n'])

week_count_data['n'] = week_count_data['n'].astype(int)  # Convert to int for better readability

# Adding a label for 'TraditionalStrengthTraining' top shorten it for the graph output
week_count_data['activity'] = week_count_data['activity'].replace({'TraditionalStrengthTraining': 'Weights'})

week_count_data = pd.merge(week_count_data,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
############### Frequency bar chart grouped by exercise type and week ###############
#week_count_data = gen_week_freq_df(aw_final, l_3_m)
#def Weekly_Freq_BarChart():

plot = (ggplot(week_count_data, aes(x='week', y='n', fill='activity')) +
    geom_bar(stat='identity', position='dodge', color = "Black") +
    geom_text(aes(label='n'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 
    
    scale_fill_brewer(type='qual', palette='Set2') +
    #scale_color_manual(values={'Running': 'Black', 'Cycling': 'Gray'}) +
    scale_y_continuous(breaks = range(0, 6),
                    limits = [0, 5]) +

    labs(title='Workout Frequency by Week and Activity',
        x='',
        y='Sessions',
        fill='Activity') +
    #theme_matplotlib() +
    theme_seaborn() +
    theme(figure_size=(10, 5)))

plot

#### Distance per week grouped by exercise type

In [ ]:
############### Distance per week grouped by exercise type ###############
# def gen_distance_df(aw_final, filter2):

miles_week = (aw_all[(aw_all['activity'].isin(['Running', 'Cycling'])) & (aw_all['metric'].str.contains("Distance")) & (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity', 'week_date'])['value']
    .agg(Total_Miles='sum', n='count')  # compute both mean and count
    .round(2) # Round to 2 decimal places
    .reset_index())

# What's interesting is that if biking or running has 0 observations missing observations won't be filled in
full_miles_week = fill_missing_combinations(
    original_df=miles_week,
    aggregated_df=miles_week,
    time_col='week_date',
    category_col='activity',
    value_cols=['Total_Miles', 'n'])


full_miles_week = pd.merge(full_miles_week,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

# Using the max point in the graph to calculate max Y axis to give enough room for horizonal legend
y_max = full_miles_week["Total_Miles"].max()

cushion = 6

y_limit = round(cushion + y_max)

In [ ]:
############### Distance per week grouped by exercise type ###############
#full_miles_week = gen_distance_df(aw_final, l_3_m)
#def Distance_BarChart():

plot = (ggplot(full_miles_week, aes(x='week', y='Total_Miles', group='activity' )) +
        
geom_line(aes( color='activity' ), size=1.2) +
geom_point(aes( color='activity' )) +

#geom_bar(stat='identity', position='dodge', color = "Black") +


geom_text(aes(label='Total_Miles'),  va='bottom') + #position=position_dodge(width=.9),

scale_y_continuous(breaks = range(0, y_limit, 5),
                #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                limits = [0, y_limit]) +

scale_color_manual(values={'Running': '#a259d9',   
    'Cycling': '#ff9800'}) +

labs(title= "",
    x="",
    y="Miles",
    fill = "Activity") +
theme_seaborn() +

theme(figure_size=(10, 5),
      legend_position= (.5, .95),
      legend_title=element_blank(),
      legend_direction='horizontal',
      legend_text=element_text(color="Black", size=12),
      legend_background= element_blank(), #element_rect(fill="#1E2A38", color=None),
      legend_key=element_blank(), # Removes boxing/shading around lines in legend
      panel_grid_minor_y = element_line(color = "White", linetype = "solid")))

plot

#### Minutes per week grouped by cardio and weights (Deprecated)

In [ ]:
############### Minutes per week grouped by cardio and weights ###############
# def gen_mins_df(aw_final, filter2):

mins_week = (aw_all[
        (aw_all['activity_type'].notna()) &
        (aw_all['metric'] == "Duration") &
        (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity_type', 'week_date'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())

full_mins_week = fill_missing_combinations(
    original_df=mins_week,
    aggregated_df=mins_week,
    time_col='week_date',
    category_col='activity_type',
    value_cols=['Total_min', 'n'])

full_mins_week = pd.merge(full_mins_week,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
############### Minutes per week grouped by cardio and weights ############### 
#full_mins_week = gen_mins_df(aw_final, l_3_m)
#def Minutes_BarChart():
plot = (ggplot(full_mins_week, aes(x='week', y='Total_min', color='activity_type', group='activity_type')) +
    geom_point(color = "Black", size = .6) +
    geom_line(size = 1.5) +

    geom_text(aes(label='n'), va='bottom', color = "black") + 
    #geom_bar(stat='identity', position='dodge', color = "Black") +
    #geom_text(aes(label='Total_min'), format_string='{:.1f}',position=position_dodge(width=.9), va='bottom') + #Format count?

    geom_hline(yintercept=150, color="#549f74", linetype='dashed', size=1) + # Cardio goal
    geom_hline(yintercept=80, color="#b36a62", linetype='dashed', size=1) + # Weights goal
    
    scale_y_continuous(breaks = range(0, y_max, 50),
                    #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
                    limits = [0, y_max]) +

    # Manual color scales
    #scale_fill_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +  # Bar colors

    scale_color_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +

    labs(title= "Minutes per Week by Activity",
        x="",
        y="Minutes", 
        fill="Activity") +
    
    theme_seaborn() +
    theme(figure_size=(10, 5),
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted"),
        legend_direction="vertical",
        legend_title=element_blank(), 
        legend_position=(.95, .92) ))

plot

#### Workout time grouped by workout type, specified with specific exercise

In [ ]:
# Calculating workout during on activity types for their underlying activities 

## Cardio df
full_mins_cardio = (aw_all[
        (aw_all['activity_type'] == "Cardio") &
        (aw_all['metric'] == "Duration") &
        (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity', 'week_date'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())


## Weight df
full_mins_weight = (aw_all[
        (aw_all['activity_type'] == "Weights") &
        (aw_all['metric'] == "Duration") &
        (aw_all['week_date'] >= l_3_m)]
    .groupby(['activity', 'week_date'])['value']
    .agg(Total_min='sum', n='count')  # compute sum and count
    .round(2)  # Round to 2 decimal places
    .reset_index())

## Changing Total_min from float to rounded int
full_mins_cardio["Total_min"] = round(full_mins_cardio["Total_min"], 0).astype("int64")

## Offsetting the weekdate so I can plot both cardio and weight bars on the same graph
full_mins_cardio['x_nudged']  = full_mins_cardio['week_date']  - timedelta(days=1)
full_mins_weight['x_nudged'] = full_mins_weight['week_date'] + timedelta(days=1)

In [ ]:
# Calculating total minutes per week for cardio workouts (used to set y axis max and label calculation)
total_mins = full_mins_cardio.groupby(['week_date'])['Total_min'].agg(week_min='sum')

# Instituing some quality of life for my legend
y_cush = 50 
y_max = int(y_cush + max(total_mins["week_min"]))

# Merging total cardio mins for each week to full_mins_cardio and renaming to cardio_labels 
cardio_labels = pd.merge(full_mins_cardio, total_mins, on= "week_date", how="left")

# Calculating the percetnage of cardio time spent on each activity for each week 
cardio_labels["percent_time"] = round(cardio_labels["Total_min"] / cardio_labels["week_min"] * 100, 2)

# Sorting cardio_labels by week_date and activity for cumsum calculation to match how activities are stacked on barplot
cardio_labels = cardio_labels.sort_values(by=['week_date', 'activity'], ascending=[True, False])
cardio_labels["cumsum_min"] = cardio_labels.groupby('week_date')['Total_min'].cumsum()

#del total_mins

In [ ]:
workout_theme = theme(
    figure_size=(10, 5),
    legend_position=(.5, .95),
    legend_title=element_text(color="#1E3A52", size=11, weight='bold'),
    legend_direction='horizontal',
    legend_text=element_text(color="#3A5A78", size=9),
    legend_background=element_blank(),
    legend_key=element_blank(),

    plot_background  = element_rect(fill="#FFFFFF", color=None),
    panel_background = element_rect(fill="#F4F7FA", color=None),
    panel_grid_major = element_line(color="#D9E4EE", size=0.5),
    panel_grid_minor_y = element_line(color="#EBF1F6", linetype="solid"),

    axis_text        = element_text(color="#6B8299", size=9),
    axis_title       = element_text(color="#3A5A78", size=10),
    plot_title       = element_text(color="#1E3A52", size=13, weight="bold"),
)

In [ ]:
############### Plot ###############
plot = (
    ggplot() +

    # Cardio stacked bar chart by activity subtypes (nudged to the left)
    geom_bar(
        data=full_mins_cardio,
        mapping=aes(x='x_nudged', y='Total_min', fill='activity'),
        stat='identity', position='stack', width=2, color='#b36a62', size=0.7 
    ) +

    # Cardio Labels: minutes for each cardio subtype
    geom_text(
        data=cardio_labels,
        mapping=aes(x='x_nudged', y='cumsum_min', label='Total_min'),
        #format_string='{:.1f}', 
        va='bottom', color='black', size=8 ) +

    # Weights barchart, not stacked because only 1 activity type (nudged right)
    geom_bar(
        data=full_mins_weight,
        mapping=aes(x='x_nudged', y='Total_min', fill='activity'),
        stat='identity', position='stack', width=2, color='#549f74', size=0.7 ) +

    # Weight labels: count of weight exercises per week
    geom_text(data = full_mins_weight,
              mapping= aes(x='x_nudged', y='Total_min', label='n'), 
              va='bottom', color = "black", size = 8) +

    # 
    scale_fill_manual(values= {"Swimming": "#00C9D4",
                            "Cycling":  "#FF6B2B",
                            "Running":  "#E63946",
                            "Walking":  "#9B8EC4",
                            "TraditionalStrengthTraining":  "#7EBC1A"},
                            labels={"TraditionalStrengthTraining": "Weights"}) +

    # Setting goal lines for cardio and weight exercises per week
    geom_hline(yintercept=150, color="#b36a62", linetype='dashed', size=1) +
    geom_hline(yintercept=80,  color="#549f74", linetype='dashed', size=1) + 

    scale_y_continuous(breaks=range(0, y_max + 1, 50), 
                       limits=[0, y_max]) +

    scale_x_datetime(date_breaks='1 week', 
                     date_labels='%b %d',
                     expand=(.01, 0)) +

    labs(title="", 
         x="", 
         y="Minutes", 
         fill="") +

    workout_theme

)

plot

#### Workout time by week over time

In [ ]:
# Gather average weekly time spent working out using last 3 months data
workout_time_df = aw_all[(aw_all['metric'] == 'Duration') & (aw_all['week_date'].dt.year >= 2024) ].groupby(['week_date'])['value'].agg(Time='sum', n='count').reset_index()  

workout_time_df["Hours"] = (workout_time_df["Time"]/60).round(2)
workout_time_df["Year"] = pd.Categorical(workout_time_df['week_date'].dt.year.astype(str))
workout_time_df["Hours"] = (workout_time_df["Time"]/60).round(2)
workout_time_df["month_day"] = workout_time_df['week_date'].dt.strftime('%m-%d')  # Extract month-day for alignment

# Create a reference date in a common year (e.g., 2024) for plotting
workout_time_df["plot_date"] = pd.to_datetime('2024-' + workout_time_df["month_day"])

# Creating normalized datetime for all years
# We set every year to 2000 so they align on the x-axis, but keep month/day
workout_time_df['norm_date'] = workout_time_df['week_date'].apply(lambda dt: dt.replace(year=2000))

# Dynamically calculating y axis max with some cushion room for labels
y_cush = 2
max_hours = workout_time_df['Hours'].max()

y_max = int(y_cush + max_hours)

plot = (ggplot(workout_time_df, aes(x= 'plot_date', y='Hours', group = 'Year', color = 'Year')) +
    geom_line(size=1.2) +
    geom_point(color = "Black", size = .8) +
    #geom_text(aes(label='Hours'), position=position_dodge(width=0.9), va='bottom') + # va & ha are used for veritcal and horizontal allignment 

    geom_hline(yintercept=3, color="#549f74", linetype='dashed', size=1) + # Cardio goal
    
    #scale_fill_brewer(type='qual', palette='Set2') +
    scale_color_manual(values={'2024': "#52be80", '2025': '#ec7063', '2026': "#3414B3"}) + 


    scale_x_datetime(date_labels='%b', date_breaks='1 month', expand=(0, 8, 0, 1)) + # https://plotnine.org/reference/scale.html 

    scale_y_continuous(breaks = range(0, y_max + 1), # Defining breaks of y axis (every number between 0 and the calculated max within the limit)
                    limits = [0, y_max]) + # Defining zoom of y axis 

    labs(title='', # Workout Duration by Week and Activity
        x='',
        y='Hours',
        color='Year') +

    #theme_seaborn() +

    workout_theme
)

plot

#### Minutes per week for all exercises (line grpah - Deprecated)

In [ ]:
############### Minutes per week for all exercises ############### 
# def gen_workout_time_df(aw_final, filter1):

workout_time = (
    aw_all[(aw_all['week_date'] > l_3_m) & (aw_all['metric'] == 'Duration')]
    .groupby('week_date', observed=True)['value']
    .agg(Time='sum', n='count')
    .round(2)
    .reset_index())

workout_time = pd.merge(workout_time,
    aw_all[['week', 'week_date']].drop_duplicates(),
    on='week_date',
    how='left')

In [ ]:
############### Minutes per week for all exercises ############### 
#workout_time = gen_workout_time_df(aw_final, l_7_m)
#def Minutes_LineGraph():

plot = (ggplot(workout_time, aes(x='week', y='Time')) +
    geom_line(color = "blue", size = 1) +
    #geom_point(aes(size = "n"), alpha = 0.6, color = "blue") +
    #geom_text(aes(label='n'), format_string='{:.0f}', va='bottom') +
    
    labs(title= "Minutes per Week by Activity",
         x="Week",
         y="Minutes") +

    #scale_x_datetime(date_labels='%b %d', date_breaks='1 week') +
    
    theme_seaborn() +
    
    theme(panel_grid_minor_y = element_line(color = "gray", linetype = "dotted"),
        figure_size=(10, 5),
        axis_text_x=element_text(angle=25, hjust=1),
        legend_position='none',
        axis_ticks_minor_x=element_blank()))

plot

#### Activity Treemap Distribution

In [ ]:
############### Activity Treemap ###############  
#def gen_activity_treemap_df(aw_all, l_3_m):

activites = ['Running', 'Cycling', 'TraditionalStrengthTraining', 'Swimming']

activity_distribution = (aw_all[
        (aw_all['metric'] == "Duration") & 
        (aw_all['activity'].isin(activites)) & 
        (aw_all['week_date'] >= l_3_m)] # Filtering by the last 3 months
    .groupby(['activity'])['value']
    .agg(n='count')
    .reset_index())
            
# Add percent of total column
total = activity_distribution['n'].sum()
activity_distribution['percent'] = activity_distribution['n'] / total

# Format labels as "Activity<br>Count (Percent)"
activity_distribution['label'] = activity_distribution.apply(lambda row: f"{row['activity']}<br>{row['n']} ({row['percent']:.1%})", axis=1)

In [ ]:
# Create treemap
#activity_distribution = gen_activity_treemap_df(aw_final, l_3_m)
#def activity_treemap():

fig = px.treemap(
    activity_distribution,
    path=['label'],         # use custom label for full text
    values='n',
    color='n',
    color_continuous_scale='Blues')

fig.update_traces(hovertemplate='%{label}')  
fig.update_layout(showlegend=False, coloraxis_showscale=False)

# treemap_plotly_html = pio.to_html(fig, full_html=False)

#### Step count grouped by month boxplot

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
engine = create_engine(f"sqlite:///habits.db")

#def gen_steps_month_df(filter3):

# Calculating steps per day in SQLite, then using localtime function to get to EST from UTC
with engine.connect() as connection:
    apple_steps = pd.read_sql_query(
        text("""SELECT type, sum(value) as 'value', date(startDate, 'localtime') as 'date', DATETIME(startDate, 'start of month') as 'month_date'
                FROM apple_data_raw
                WHERE type = 'StepCount' AND value IS NOT NULL and date(startDate, 'start of month','+1 month','-1 day') >= :t_filter
                GROUP BY date(startDate)
                ORDER BY 'date'"""), connection,
        dtype={"value": "int64"},
        params={"t_filter": l_1_y.astimezone(ZoneInfo("UTC"))},
        parse_dates=['date', 'month_date'])

# Localizing date and month_date to US/Eastern because already convert to localtime in the SQL query
apple_steps['date'] = apple_steps['date'].dt.tz_localize("US/Eastern")
apple_steps['month_date'] = apple_steps['month_date'].dt.tz_localize("US/Eastern")

# Creating a month categorical variable
apple_steps['month'] = apple_steps['date'].dt.strftime('%b %Y') # Need .dt for series/panda

# Set 'month' as a categorical(factor variable) with order based on 'month_date'
month_order = apple_steps.sort_values('month_date')['month'].unique()
apple_steps['month'] = pd.Categorical(apple_steps['month'], categories=month_order, ordered=True)

# Dynamically calculating y axis max with some cushion room for labels
step_cush = 2000

step_y_max = int(apple_steps["value"].max() + step_cush)

# apple_steps = gen_steps_month_df(l_1_y)

In [ ]:
apple_steps

In [ ]:
############### Step count grouped by month ############### 
#steps_day = gen_steps_month_df(l_1_y)

#def Steps_Boxplot():
plot = (
    ggplot(apple_steps, aes(x='month', y='value', fill='month')) +

    geom_boxplot(color="black") +
    stat_summary(fun_data="mean_cl_boot", geom = "point", fill = "white", color = "red") +

    geom_hline(yintercept=5000, color="#549f74", linetype='dashed', size=1) + # Step goal

    labs(title="", 
        x="", 
        y="Steps") +

    scale_y_continuous(breaks = range(0, step_y_max , 5000),
                    minor_breaks=range(0, step_y_max , 2500),  # Can't do minor ticks 2.5 because int not float :(
                    limits = [0, step_y_max]) +

    theme_seaborn() +

    theme(axis_text_x=element_text(),
        figure_size=(10, 5),
        legend_position='none',
        panel_grid_minor_y = element_line(color = "White", linetype = "solid")))
plot

### HTML Tables

#### Filtered Exercise Table

In [ ]:
# Generates a table for the dynamic exercise filter page 
#def specific_exercise_filter(specific_exercise):

specific_exercise = "Chest press machine"  # Simulating user input
columns = ['entry_id', '10RM', 'comment', 'effort', 'reps', 'sets', 'weight', 'workout_type']
workout_df_total = pd.DataFrame(columns=columns)

with engine.connect() as connection:

    entry_ids = pd.read_sql_query(text("SELECT ha.entry_id FROM habit_answers ha WHERE answer = :exercise"), connection, params={"exercise": specific_exercise})

    entry_ids = entry_ids['entry_id'].tolist()


    for id in entry_ids:
        workout_df = pd.read_sql_query(text("SELECT entry_id, question, answer FROM habit_answers ha WHERE entry_id = :id"), connection, params={"id": id})

        workout_df = workout_df.pivot(index='entry_id', columns='question', values='answer').reset_index() 

        workout_df_total = pd.concat([workout_df_total, workout_df], ignore_index=True)

    habit_entries = pd.read_sql_query(text("SELECT log_date, id FROM habit_entries"), connection)

    # Merging and selecting relevant columns
    workout_df_total = pd.merge(workout_df_total, habit_entries, left_on='entry_id', right_on='id', how='left')[["log_date", "workout_type", "weight", "sets", "reps", "effort", "comment"]]

    workout_df_total['comment'] = workout_df_total['comment'].str.replace("nan", "")

    workout_df_total.rename(columns={'log_date': 'Timestamp', 'workout_type': 'Exercise', 'weight': 'Weight', 'sets': 'Sets', 'reps': 'Reps',
                             'effort': 'Effort Level', 'comment': 'Notes:'}, inplace=True)


    # reading in 10RM workouts to add 
    ten_rm_additions = pd.read_sql_query(
        text("""SELECT tc.completion_date as Timestamp, tp.exercise_name as Exercise, tp.target_weight as Weight, tp.sets as Sets, tp.reps as Reps, tc.notes as 'Notes:' 
                FROM tenrm_completions tc
                LEFT JOIN tenrm_plans tp 
                    ON tp.id = tc.plan_id
                WHERE tp.exercise_name = :exercise
                AND tc."timestamp" = (
                        SELECT MAX(tc2."timestamp")
                        FROM tenrm_completions tc2
                        -- Correlated Subquery
                        -- This works because we loop through plan_ids in orig table until it equals max plan_id
                        WHERE tc2.plan_id = tc.plan_id)"""), connection, params={"exercise": specific_exercise})

    # Adding in effort level as a blank variable since 10rm data doesn't track that
    ten_rm_additions['Effort Level'] = ""

    # Combining original workouts with 10rm workouts (Since they have the same columns and format) 
    combined_works = pd.concat([workout_df_total, ten_rm_additions], ignore_index=True) 

    # Convert the timestamp variable to a datetime format ( Could have done this sql query with parse dates too)
    combined_works['Timestamp'] = pd.to_datetime(combined_works['Timestamp'])

    # Sort the combined_works table by date before converting to string output (for readability)
    combined_works = combined_works.sort_values('Timestamp')

    # Converting Timestamp into readable string format (flexability to display time however I want
    combined_works['Timestamp'] = combined_works['Timestamp'].dt.strftime('%B %d, %Y')

#return combined_works.to_html(classes='workout-table', index=False, border=1)

### KPI Calculations

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

############### KPI Statisitcs Functions ###############

# Going to have to recreate this function with apple workout data
#def get_kpi_stats(apple_steps, l_3_m):

# Calculating steps per day in SQLite, then using localtime function to get to EST from UTC
with engine.connect() as connection:
    daw = pd.read_sql_query( # daw = distinct apple workouts
        text("""SELECT DATE(aw.startDate, 'localtime') as Date, aw.activity, aw.value 
                FROM apple_workouts aw 
                WHERE aw.metric = 'Duration' AND aw.value > 10
                ORDER BY aw.StartDate desc """), connection,
        #dtype={"value": "int64"},
        #params={"t_filter": l_1_y.astimezone(ZoneInfo("UTC"))},
        parse_dates=['Date'])
    
# Creating a week categorical variable (copied from inital aw_all completion)
# calculating out necessary time metrics when reading in apple workouts data
## calculates how many days each weekday is from monday and subtracts that from the orignal date to get back to start of the week Monday
daw['week_date'] = (daw['Date'] - pd.to_timedelta(daw['Date'].dt.weekday, unit="D") ).dt.normalize() # Resulting date is in EST (I believe this is the correct move when filtering later on) # dt.weekday uses Monday as 0 

    # Create a string label for display
daw['week'] = daw['week_date'].dt.strftime('%b %d')

# Set 'week_label' as a categorical(factor variable) with order based on 'week_period'
week_order = daw.sort_values('week_date')['week'].unique()
daw['week'] = pd.Categorical(daw['week'], categories=week_order, ordered=True)


# Gathering times for filtering
today = datetime.now(tz=ZoneInfo("US/Eastern"))
this_month = today.strftime('%b %Y')
last_month = (today - relativedelta(months=1)).strftime('%b %Y')
current_year = int(today.strftime('%Y'))
duration_length = 10

# Gather names for displaying on front end
current_month_name = today.strftime('%B')
last_month_name = (today - relativedelta(months=1)).strftime('%B') 

# Gathering wokrout counts (classifying a true workout to be greater than 10 minutes long)
## Current Month Workout Count
wokrout_count_CM = len( daw[ (daw['Date'].dt.strftime('%b %Y') == this_month) ] )
## Last month Workout Count
workout_count_LM = len( daw[ (daw['Date'].dt.strftime('%b %Y') == last_month) ] )
## Current Year Workout Count
workout_count_year = len( daw[ (daw['Date'].dt.year == current_year) ] )

# Gather average weekly time spent working out using last 3 months data
workout_time_df = daw[ (daw['Date'].dt.date > l_3_m.date() ) ].groupby(['week_date'])['value'].agg(Time='sum', n='count').reset_index()
workout_time_hrs_avg = (workout_time_df["Time"].mean()/60).round(2) # Converting minutes to hours


# Calculating steps per day in SQLite, then using localtime function to get to EST from UTC
with engine.connect() as connection:
    das_l3m = pd.read_sql_query( # das_l3m = daily average steps last 3 months
        text("""SELECT AVG(value) as 'value'
				FROM (SELECT DATE(adr.startDate, 'localtime') as Date, SUM(adr.value) as 'value' 
						FROM apple_data_raw adr
						WHERE type = 'StepCount' AND value IS NOT NULL AND date(startDate, 'localtime') >= :t_filter
						GROUP BY date(startDate, 'localtime')
						) """), connection,
        #dtype={"value": "int64"},
        params={"t_filter": l_3_m} )   #.astimezone(ZoneInfo("UTC"))},
        #parse_dates=['Date'])

steps_L3_mon = round(das_l3m["value"].iloc[0], 0).astype("int")


# Saving output to csv instead of directly connecting to flask
asaved_df = pd.DataFrame({
    'workout_count_year': workout_count_year,
    'workout_count_LM': workout_count_LM,
    'workout_count_CM': wokrout_count_CM,
    'current_month_name': current_month_name,
    'last_month_name': last_month_name,
    'workout_time_hrs_avg': workout_time_hrs_avg,
    'steps_L3_mon': steps_L3_mon
}, index=[0])

asaved_df.to_csv('kpi_stats.csv', index=False)

In [ ]:
# Reading in saved csv data for kpis into main.py

import pandas as pd
df = pd.read_csv(r'static\charts\kpi_stats.csv')

workout_count_year = df['workout_count_year'].iloc[0]
workout_count_LM = df['workout_count_LM'].iloc[0]
workout_count_CM = df['workout_count_CM'].iloc[0]
current_month_name = df['current_month_name'].iloc[0]
last_month_name = df['last_month_name'].iloc[0]
workout_time_hrs_avg = df['workout_time_hrs_avg'].iloc[0]
steps_L3_mon = df['steps_L3_mon'].iloc[0]


#### Average daily steps

In [ ]:
# Calculating avg daily steps last 2 weeks
steps_L2_week = apple_steps[apple_steps['date'] >= l_2_w]['value'].mean().round(0).astype("int")

## Last 2 weeks steps are ___ % (higher/lower) than last 3 months
steps_pct_diff = (steps_L2_week - steps_L3_mon) / steps_L3_mon * 100

# Need to figure out what metrics are going to show arrow

#### Number of weekly workouts

#### Calories burned per day

In [ ]:
# Rn this aggregates active and passive calories, could change to just look at active calories burned
with engine.connect() as connection:
    apple_calories = pd.read_sql_query(
        text("""SELECT sum(value) as 'value', date(startDate, 'localtime') as 'date', DATETIME(startDate, 'start of month') as 'month_date' -- type
                                FROM apple_data_raw
                                WHERE type in ('BasalEnergyBurned', 'ActiveEnergyBurned') AND value IS NOT NULL and date(startDate) >= :t_filter
                                GROUP BY date(startDate)
                                ORDER BY 'date' desc"""), connection,
        dtype={"value": "int64"},
        params={"t_filter": l_3_m.astimezone(ZoneInfo("UTC"))},
        parse_dates=['date', 'month_date'])

# Localizing date and month_date to US/Eastern because already convert to localtime in the SQL query
apple_calories['date'] = apple_calories['date'].dt.tz_localize("US/Eastern")
apple_calories['month_date'] = apple_calories['month_date'].dt.tz_localize("US/Eastern")

# Creating a month categorical variable
apple_calories['month'] = apple_calories['date'].dt.strftime('%b %Y') # Need .dt for series/panda
# Set 'month' as a categorical(factor variable) with order based on 'month_date'
month_order = apple_calories.sort_values('month_date')['month'].unique()
apple_calories['month'] = pd.Categorical(apple_calories['month'], categories=month_order, ordered=True)

In [ ]:
calories_l_3_m = apple_calories["value"].mean().round(0).astype("int")

calories_l_2_w = apple_calories[apple_calories['date'] >= l_2_w]['value'].mean().round(0).astype("int")

## Last 2 weeks steps are ___ % (higher/lower) than last 3 months
calorie_pct_diff = (calories_l_2_w - calories_l_3_m) / calories_l_3_m * 100

#### Average weekly workout time

In [ ]:
# Average weekly time spent working out using last 2 weeks query

workout_time_l2w = (aw_all[(aw_all['metric'] == 'Duration') &
                             (aw_all['week_date'] > l_2_w)]
                      .groupby(['week_date'])['value']
                      .agg(Time='sum', n='count')
                      .reset_index()['Time']
                      .mean() / 60).round(2)

# Last 3 months already calculated
print(workout_time_l3m)

## Options list (workouts and books)

In [ ]:
# List to log existing workouts from db
def load_workout_options(category):

    with engine.connect() as connection:
        workouts = pd.read_sql_query(
            text("""SELECT DISTINCT answer FROM habit_answers
                    WHERE question = 'workout_type'
                    ORDER BY answer"""), connection)
        
    workouts_list = workouts['answer'].tolist()

    # Adding some logic to use this function multiple places depending on whether I need an "Other" option or not 
    if category == 0:
        pass
    elif category == 1:
        # Add "Other" option to the list of book titles and works. This way other is not saved to the database
        workouts_list.append("Other")    


    return workouts_list

In [ ]:
# Update the list of book titles table in my database with any potential new entries
def load_book_options():

    # read_sql_query simple 
    with engine.connect() as connection:
        books_options = pd.read_sql_query(
            text("""SELECT DISTINCT answer FROM habit_answers
                    WHERE question = 'book_title'
                    ORDER BY answer"""), connection)
        
    book_titles = books_options["answer"].to_list()

    # Add an "Other" option to the list of book titles because 'other' is not saved to the database
    book_titles.append("Other") 

    return book_titles

# DB.py

## Creating DB connection and modifiers

In [ ]:
# db.py
from sqlalchemy import create_engine, event, text

# For SQLite file-based DB
DATABASE_URL = "sqlite:///habits.db"

# Create a single shared engine
engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,   # ensures dead connections are detected/recycled
    connect_args={"check_same_thread": False}  # needed for SQLite in multi-threaded apps
)

# Enable WAL mode on every new connection
@event.listens_for(engine, "connect")
def set_sqlite_pragma(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute("PRAGMA journal_mode=WAL")   # Enable Write-Ahead Logging
    cursor.execute("PRAGMA synchronous=FULL") # balance durability vs performance 
    cursor.execute("PRAGMA optimize") 
    cursor.close()


## Creating sqlite tables for app

In [ ]:
# initalizing habits websites with sql tables needed to support it
# I've already gone through this and changed types to be sqlite compliant
def init_db():
    with engine.begin() as conn:

        # habits table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habits (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            name TEXT UNIQUE NOT NULL);"""))

        # habit_entries table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_entries (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            habit_id INTEGER NOT NULL,
                            log_date TEXT NOT NULL,
                            timestamp TEXT NOT NULL,
                            FOREIGN KEY (habit_id) REFERENCES habits(id));"""))

        # habit_answers table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_answers (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            entry_id INTEGER NOT NULL,
                            question TEXT NOT NULL,
                            answer TEXT NOT NULL,
                            FOREIGN KEY (entry_id) REFERENCES habit_entries(id));"""))
        
        # access log table (for IP address logging)
        conn.execute(text("""CREATE TABLE IF NOT EXISTS access_log (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            ip TEXT NOT NULL,
                            endpoint TEXT NOT NULL,
                            timestamp TEXT NOT NULL);"""))
        
        # 10RM workout plans table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS tenrm_plans (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            workout_type TEXT NOT NULL,
                            week_number INTEGER NOT NULL,
                            exercise_name TEXT NOT NULL,
                            target_weight INTEGER NOT NULL,
                            sets INTEGER NOT NULL,
                            reps INTEGER NOT NULL,
                            created_date TEXT NOT NULL,
                            UNIQUE(workout_type, week_number, exercise_name));"""))
        
        # 10RM workout completions table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS tenrm_completions (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            plan_id INTEGER NOT NULL,
                            completion_date TEXT NOT NULL,
                            completed INTEGER NOT NULL,
                            notes TEXT,
                            timestamp TEXT NOT NULL,
                            FOREIGN KEY (plan_id) REFERENCES tenrm_plans(id));"""))

        # apple_data_raw table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS apple_data_raw (
                            type TEXT, 
                            "sourceName" TEXT, 
                            value TEXT, 
                            unit TEXT, 
                            "startDate" TEXT, 
                            "endDate" TEXT, 
                            "creationDate" TEXT, 
                            "sourceVersion" TEXT, 
                            "appleStandHours" REAL, 
                            "appleExerciseTimeGoal" REAL, 
                            bpm REAL, 
                            maximum REAL, 
                            "sum" REAL, 
                            "appleMoveTimeGoal" REAL, 
                            "average" REAL, 
                            time TEXT, 
                            "key" TEXT, 
                            duration REAL, 
                            "dateComponents" TEXT, 
                            "CardioFitnessMedicationsUse" TEXT, 
                            "activeEnergyBurned" REAL, 
                            "appleMoveTime" REAL, 
                            date TEXT, 
                            "activeEnergyBurnedUnit" TEXT, 
                            locale TEXT, 
                            "appleStandHoursGoal" REAL, 
                            "BiologicalSex" TEXT, 
                            "FitzpatrickSkinType" TEXT, 
                            "BloodType" TEXT, 
                            "workoutActivityType" TEXT, 
                            "minimum" REAL, 
                            path TEXT, 
                            "appleExerciseTime" REAL, 
                            "durationUnit" TEXT, 
                            "DateOfBirth" TEXT, 
                            device TEXT, 
                            "activeEnergyBurnedGoal" REAL)"""))

        # Cleaned Apple Workouts table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS apple_workouts (
                            "StartDate" TEXT, 
                            activity TEXT, 
                            metric TEXT, 
                            measurement_type TEXT, 
                            value REAL, 
                            d_unit TEXT, 
                            activity_type TEXT, 
                            workout_id REAL)"""))
        
                # Bike Workout table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS bike_workout (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            start_time TEXT,
                            end_time TEXT,
                            upload_date TEXT)"""))
        
        # Bike Units table (reference table - no foreign keys needed)
        conn.execute(text("""CREATE TABLE IF NOT EXISTS bike_units (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            metric_type TEXT UNIQUE NOT NULL,
                            metric_unit TEXT NOT NULL,
                            unit_name TEXT NOT NULL)"""))
        
        # Bike Workout Metrics table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS bike_metrics (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            timestamp TEXT NOT NULL,
                            workout_id INTEGER NOT NULL,
                            metric_id INTEGER NOT NULL,
                            value REAL,
                            FOREIGN KEY (workout_id) REFERENCES bike_workout(id),
                            FOREIGN KEY (metric_id) REFERENCES bike_units(id))"""))

# Main.py

In [ ]:
########### Log users IP address after every made request
def get_client_ip():
    """Helper to get the client IP address, accounting for proxies."""
    if request.headers.get("X-Forwarded-For"):
        # Might contain multiple IPs, take the first one
        return request.headers.get("X-Forwarded-For").split(",")[0].strip()
    return request.remote_addr or "unknown"

@app.before_request
def log_ip():
    ip = get_client_ip()
    endpoint = request.path
    today = datetime.now(tz=ZoneInfo("US/Eastern")).strftime("%Y-%m-%d %H:%M:%S")

    with engine.begin() as conn:
        conn.execute(
            text("""INSERT INTO access_log (ip, endpoint, timestamp)
                    VALUES (:ip, :endpoint, :timestamp)"""),
            {"ip": ip, "endpoint": endpoint, "timestamp": today},
        )


## 10RM Functions

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# For SQLite file-based DB
DATABASE_URL = "sqlite:///habits.db"

# Create a single shared engine
engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,   # ensures dead connections are detected/recycled
    connect_args={"check_same_thread": False}  # needed for SQLite in multi-threaded apps
)

########### 10 RM workouts page ###########
#@app.route('/10rm_tracker', methods=['GET'])
#def tenrm_tracker():

# Read in the data and parse datetime columns
with engine.connect() as connection:
    all_10rms = pd.read_sql_query(
        text("""
                SELECT tp.id, tp.workout_type, tp.week_number, tp.exercise_name, tp.target_weight, tp.sets, tp.reps, tp.created_date, tc.completion_date, tc.completed, tc.notes, tc.id as tc_id
                FROM tenrm_plans tp
                LEFT JOIN tenrm_completions tc
                ON tp.id = tc.plan_id
                ORDER BY tp.workout_type, tp.week_number, tp.exercise_name
             """), connection,
    parse_dates=["created_date", "completion_date"])


    workout_type_dates = pd.read_sql_query(
        text("""
            SELECT tp.workout_type, MAX(tp.created_date) as max_date
            FROM tenrm_plans tp
            GROUP BY tp.workout_type
             """), connection,
    parse_dates=["max_date"])


all_10rms = pd.merge(all_10rms, workout_type_dates, on='workout_type', how='left')
all_10rms = all_10rms[all_10rms['created_date'] == all_10rms['max_date']]


all_10rms = (
    all_10rms
    .sort_values("tc_id")
    .groupby("tc_id", dropna=False)
    .last()
    .reset_index()
)

# Organize data by workout type, week, and created date
organized_data = {}



In [ ]:
#    return render_template('10rm_tracker.html', workout_data=organized_data)

for _, row in all_10rms.iterrows():

    workout_type = row['workout_type']
    week_number = row['week_number']

    if workout_type not in organized_data:
        organized_data[workout_type] = {}

    if week_number not in organized_data[workout_type]:
        organized_data[workout_type][week_number] = []

    organized_data[workout_type][week_number].append({
        'plan_id': row['id'],
        'exercise_name': row['exercise_name'],
        'target_weight': row['target_weight'],
        'sets': row['sets'],
        'reps': row['reps'],
        'completion_date': row['completion_date'],
        'completed': row['completed'],
        'notes': row['notes']
    })


In [ ]:
### Logging 10 RM exercises I've completed from the plan page
@app.route('/log_10rm_completion', methods=['POST'])
def log_10rm_completion():
    """Log completion of a 10RM workout"""
    try:
        data = request.json
        plan_id = data.get('plan_id')
        completion_date = data.get('completion_date')
        completed = data.get('completed')
        notes = data.get('notes', '')
        
        ########## FIX TIMEZONE ##########
        today = datetime.now(tz=ZoneInfo("US/Eastern")).replace(microsecond=0)
        
        with engine.begin() as conn:
            conn.execute(text("""
                INSERT INTO tenrm_completions (plan_id, completion_date, completed, notes, timestamp)
                VALUES (:plan_id, :completion_date, :completed, :notes, :timestamp)"""), 
            {
                'plan_id': plan_id,
                'completion_date': completion_date,
                'completed': completed,
                'notes': notes,
                'timestamp': today
            })
        
        return jsonify({'status': 'success', 'message': 'Completion logged successfully'})
    
    except Exception as e:
        return jsonify({'status': 'error', 'message': str(e)}), 500


# Bike .Fit files

## Visualizations

### Combining bike data with apple workout raw

In [ ]:
import pandas as pd
from sqlalchemy import text, create_engine

engine = create_engine(f"sqlite:///habits.db")

# Grabbing all the metric types from the metrics table which I'll use to filter incoming bike data and insert into bike_metrics
with engine.begin() as conn:
    bike_power = pd.read_sql_query(text("""SELECT bm."timestamp", bm.value as power_value, bu.metric_type
                                             FROM bike_metrics bm
                                             left outer join bike_units bu 
                                             on bu.id = bm.metric_id
                                             WHERE bu.metric_type = 'power' and bm.workout_id = 2"""), conn,
                                       parse_dates=["timestamp"],
                                       dtype={"power_value": "Int64"})
    
bike_power["timestamp"] = bike_power["timestamp"].dt.tz_localize("UTC")

In [ ]:
with engine.begin() as conn:
    workout_dates = pd.read_sql_query(
        text("""SELECT start_time, end_time 
                FROM bike_workout
                where id =2"""), conn)

In [ ]:
with engine.begin() as conn:
    apple_heart = pd.read_sql_query(text("""SELECT adr.startDate as timestamp, adr.type, adr.unit, adr.value as heart_rate_value
                                            FROM apple_data_raw adr 
                                            WHERE adr.type = 'HeartRate' and adr.startDate between :lower_date and :upper_date """), conn,    # '2025-07-12 12:33:01.000000' and '2025-07-12 13:11:56.000000'
                                        params= {"lower_date": str(workout_dates["start_time"][0]), "upper_date": str(workout_dates["end_time"][0]) },
                                       parse_dates=["timestamp"])
    
apple_heart["timestamp"] = apple_heart["timestamp"].dt.tz_localize("UTC")

In [ ]:
# Import to merge apple data into bike data because bike data is more complete for each second of workout 
combined = pd.merge(bike_power, apple_heart, on="timestamp", how="left")

combined = combined.ffill()

In [ ]:
combined_long = pd.melt(combined, id_vars=["timestamp"], value_vars=["power_value", "heart_rate_value"], var_name="metric_type", value_name="value")

combined_long["value"] = combined_long["value"].astype("float")

In [ ]:
from plotnine import ggplot, aes, geom_bar, geom_text, position_dodge, scale_fill_brewer, scale_y_continuous, labs, theme_seaborn, theme, scale_fill_manual, element_line, geom_hline, geom_line, \
geom_point, scale_x_datetime, element_text, element_blank, geom_boxplot, stat_summary, scale_color_manual, geom_smooth


plot = (ggplot(combined_long, aes(x='timestamp', y='value', color='metric_type', group='metric_type')) +
    geom_point(color = "Black", size = .6) +
    geom_line(size = 1) +

    geom_smooth(aes(y="value")) + 
    
    #scale_y_continuous(breaks = range(0, 450, 50),
    #                #minor_breaks=range(0, 36, 2),  # Can't do minor ticks 2.5 because int not float :(
    #                limits = [0, 400]) +

    # Manual color scales
    #scale_fill_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +  # Bar colors

    #scale_color_manual(values={'Cardio': '#52be80', 'Weights': '#ec7063'}) +

    labs(title= "Minutes per Week by Activity",
        x="",
        y="Value", 
        fill="Activity") +
    
    theme_seaborn() +
    theme(figure_size=(10, 5),
        panel_grid_minor_y = element_line(color = "White", linetype = "dotted")))

plot

### Looking at units df

In [ ]:
# How we handle metric units (later)
rows = []

for record in fitfile.get_messages('record'):
    row = {}

    for field in record:
        row[field.name] = field.value
        if field.units:
            row[f"{field.name}_units"] = field.units

    rows.append(row)

df = pd.DataFrame(rows)

## Cheatsheet (hidden)

In [ ]:
import markdown

cheatsheet_path = "static/coding_cheatsheet.md"

with open(cheatsheet_path, "r", encoding="utf-8") as f:
    md_text = f.read()

# Converting markdown file to HTML (actually works pretty well!)
html_cheatsheet = markdown.markdown(md_text)


In [ ]:
html_cheatsheet

## Calendar Code

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import numpy as np
import calendar
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Used to find default month and year
today = datetime.now(tz=ZoneInfo("EST")).date()

year  = today.year
month = today.month

days_in_month = calendar.monthrange(year, month)[1]

month_start = f"{year}-{month:02d}-01" # First day of month YY-MM-DD
month_end   = f"{year}-{month:02d}-{days_in_month}" # Last day of month YY-MM-DD

with engine.begin() as conn:
    mon_habits_df = pd.read_sql("""SELECT he.log_date,  h.name as habit_type, ha.entry_id, ha.question, ha.answer
                        FROM habit_answers ha
                        LEFT OUTER JOIN habit_entries he
                        ON he.id = ha.entry_id
                        LEFT OUTER JOIN habits h
                        ON h.id = he.habit_id
                        WHERE h.name in ('stretch', 'reading') AND ha.question in ('stretch_type', 'book_title') AND he.log_date BETWEEN :start_mon AND :end_mon
                        ORDER BY date(he.log_date ) DESC """, conn,
                    params= {"start_mon": month_start, "end_mon": month_end },
                    parse_dates=["log_date"])

#mon_habits_df["display_name"] = np.where(mon_habits_df["habit_type"] == "reading", "reading",
#                                 np.where(mon_habits_df["habit_type"] == "stretch", mon_habits_df["answer"], None))

#mon_habits_df["display_name"] = mon_habits_df["display_name"].str.title()

In [ ]:
day_date = datetime(2026, 4, 2)

day_df = mon_habits_df[mon_habits_df["log_date"] == day_date]

habits_logged = []
for habit_type, group in day_df.groupby("habit_type"):

    if habit_type == "reading":
        book    = group.loc[group["question"] == "book_title",  "answer"].values
        #pages   = group.loc[group["question"] == "pages",       "answer"].values
        tooltip = f"{book[0]}: page_holder"

    elif habit_type == "stretch":
        stretches = group.loc[group["question"] == "stretch_type", "answer"].values
        #times     = group.loc[group["question"] == "time",         "answer"].values

        tooltip = ""
        for stretch in stretches:
            #time = times[i] if i < len(times) else "?"
            tooltip += f"{stretch}: time\n"

    else:
        tooltip = habit_type.title()

    habits_logged.append({
        "habit_type":   habit_type,
        "display_name": habit_type.title(),
        "tooltip":      tooltip,
    })

In [ ]:
calendar_data = {}

for day in range(1, days_in_month + 1):
    day_date   = datetime(year, month, day)
    is_future  = day_date.date() > today
    habits_logged = mon_habits_df[mon_habits_df["log_date"] == day_date]["display_name"].unique().tolist()
    #habits_logged = hello.loc[hello["log_date"] == day_date, "display_name"].unique().tolist()
    habits_logged = []
    for habit_type, group in day_df.groupby("habit_type"):

        if habit_type == "reading":
            book    = group.loc[group["question"] == "book_title",  "answer"].values
            #pages   = group.loc[group["question"] == "pages",       "answer"].values
            tooltip = f"{book[0]}: page_holder"

        elif habit_type == "stretch":
            stretches = group.loc[group["question"] == "stretch_type", "answer"].values
            #times     = group.loc[group["question"] == "time",         "answer"].values

            tooltip = ""
            for stretch in stretches:
                #time = times[i] if i < len(times) else "?"
                tooltip += f"{stretch}: time\n"

        else:
            tooltip = habit_type.title()

        habits_logged.append({
            "habit_type":   habit_type,
            "display_name": habit_type.title(),
            "tooltip":      tooltip,
        })

    count = len(habits_logged)

    # Calculating Status
    if is_future:
        status = "future"
    elif count == 0:
        status = "missed"
    elif count >= 3:
        status = "completed"
    else:
        status = "partial"

    calendar_data[day] = {
        "habits": habits_logged,
        "status": status,
        "is_future": is_future,
        }



In [ ]:
#cats: day: int,  habits: list, is_future: bool, status:cat

#    if is_future:
#        status = "future"
#    elif count == 0:
#        status = "missed"
#    elif count >= 3:
#        status = "completed"
#    else:
#        status = "partial"



In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import numpy as np
import calendar
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Used to find default month and year
today = datetime.now(tz=ZoneInfo("EST")).date()

year  = today.year
month = today.month

# Clamp month to 1–12 in case of edge values (probaby not necessary)
if month < 1:  month = 1
if month > 12: month = 12

# monthrange returns (first_weekday, days_in_month) and we only care about days in month
days_in_month = calendar.monthrange(year, month)[1]

# Aligning weeks to match static calendar in html where we start on Monday
first_weekday_mon = calendar.weekday(year, month, 1)   # 0=Mon

# calendar.weekday() returns 0=Mon, but if we want 0=Sun
# so shift: Mon->1, Tue->2 ... Sat->6, Sun->0
#first_weekday_sun = (first_weekday_mon + 1) % 7    # 0=Sun

# Prev / next month for nav arrows
if month == 1:
    prev_year, prev_month = year - 1, 12
else:
    prev_year, prev_month = year, month - 1

if month == 12:
    next_year, next_month = year + 1, 1
else:
    next_year, next_month = year, month + 1

# Querying relevant habits for specific year and month given

## Building month and year string based on given month and year to filter habit data to specific month
## Formatting string for month ensures there's a leading 0 for months 1-9
month_start = f"{year}-{month:02d}-01"
month_end   = f"{year}-{month:02d}-{days_in_month}"


with engine.connect() as conn:
    rows = conn.execute(text("""
        SELECT he.log_date, h.name AS habit_name
        FROM habit_entries he
        JOIN habits h ON h.id = he.habit_id
        WHERE he.log_date BETWEEN :start AND :end
        ORDER BY he.log_date
    """), {"start": month_start, "end": month_end}).fetchall()

# Total distinct habits that exist (for "all completed" check)
# Not sure if this is calculating correctly based on off the json blob
# And would want to tailor this to specific habits of importance
#total_habits = len(HABITS)

# Build calendar_data dict  {day_int: {status, habits: [...]}}
from collections import defaultdict
day_habits = defaultdict(set)   # day -> set of habit names logged

for log_date, habit_name in rows:
    # log_date may come back as a string "YYYY-MM-DD" or a date object
    if isinstance(log_date, str):
        day = int(log_date.split("-")[2])
    else:
        day = log_date.day
    day_habits[day].add(habit_name)

calendar_data = {}
for day in range(1, days_in_month + 1):
    day_date   = datetime(year, month, day).date()
    is_future  = day_date > today
    habits_logged = sorted(day_habits.get(day, []))
    count = len(habits_logged)

    if is_future:
        status = "future"
    elif count == 0:
        status = "missed"
    elif count >= 3:
        status = "completed"
    else:
        status = "partial"

    calendar_data[day] = {
        "habits":    habits_logged,
        "status":    status,
        "is_future": is_future,
    }

# Summary stats (past days only)
days_completed    = sum(1 for d in calendar_data.values() if d["status"] == "completed")
days_partial      = sum(1 for d in calendar_data.values() if d["status"] == "partial")
days_missed       = sum(1 for d in calendar_data.values() if d["status"] == "missed")
total_habits_logged = sum(len(d["habits"]) for d in calendar_data.values())

In [ ]:
calendar_data

# Food analysis

In [1]:
from sqlalchemy import create_engine, text

# For SQLite file-based DB
DATABASE_URL = "sqlite:///habits.db"

# Create a single shared engine
engine = create_engine(
    DATABASE_URL)

# Create food day table
with engine.begin() as conn:
        conn.execute(text("""CREATE TABLE IF NOT EXISTS food_daily (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            completed TEXT,
                            food_date TEXT,
                            upload_date TEXT)"""))
        
# Create fact food table
with engine.begin() as conn:
        conn.execute(text("""CREATE TABLE IF NOT EXISTS fact_food (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            food_category TEXT, 
                            value INTEGER,
                            food_daily_id INTEGER,
                            FOREIGN KEY (food_daily_id) REFERENCES food_daily(id) )"""))

In [2]:
import pandas as pd
from datetime import datetime, date

food = pd.read_csv("dailysummary.csv")

food["Date"] = pd.to_datetime(food["Date"])

# Grabbing the lastest food date
with engine.connect() as connection: 
    latest_food_date = pd.read_sql_query( 
        text("""SELECT MAX(food_date) as max_date 
        FROM food_daily
            """), connection,                    
        parse_dates=['max_date'])
    
latest_food_date = latest_food_date["max_date"].iloc[0]

food[food["Date"] > latest_food_date]

,Date,Energy (kcal),Alcohol (g),Caffeine (mg),Oxalate (mg),Phytate (mg),Water (g),B1 (Thiamine) (mg),B2 (Riboflavin) (mg),B3 (Niacin) (mg),...,Leucine (g),Lysine (g),Methionine (g),Phenylalanine (g),Protein (g),Threonine (g),Tryptophan (g),Tyrosine (g),Valine (g),Completed


In [9]:
import pandas as pd
from datetime import datetime, date

food = pd.read_csv("dailysummary.csv")

food["Date"] = pd.to_datetime(food["Date"])

# Grabbing the lastest food date
with engine.connect() as connection: 
    latest_food_date = pd.read_sql_query( 
        text("""SELECT MAX(food_date) as max_date 
        FROM food_daily
            """), connection,                    
        parse_dates=['max_date'])
    
latest_food_date = latest_food_date["max_date"].iloc[0]

food_filt = food[food["Date"] > latest_food_date]

# Selecting specific columns then filtering by date for new uploads 
food_daily = food_filt[["Date", "Completed"]].rename(columns={"Date": "food_date", "Completed": "completed"})     

today_upload_date = date.today().strftime("%m-%d-%Y")

food_daily["upload_date"] = today_upload_date 

food_daily.to_sql(name="food_daily", con=engine, if_exists='append', index=False)

7

In [13]:
# read back in the data we just uploaded
with engine.connect() as connection: 
    # renaming food_date back to date for simplicity of merging in IDs
    # and id as food_daily_id to match with fact food table
    new_food = pd.read_sql_query( 
        text("""SELECT id as food_daily_id, food_date as Date 
        FROM food_daily
        WHERE upload_date = :today_date"""), # Kinda odd looks like max_UTC_date_db gets carried over into apple_raw (shouldn't matter though
        connection,                     # because we're only fitering to actual workouts)
        params={"today_date": str(today_upload_date)}, # Using the str version of the datetime for filtering
        parse_dates=['Date'],
        dtype={"food_daily_id": "Int64"}) # Parse date here takes out need for pd.to_datetime below 

In [14]:
# Create fact table for food
food_filt = food.drop(columns=["Completed"])

food_long = pd.melt(food_filt, id_vars=["Date"], var_name="food_category", value_name="value")

fact_food = pd.merge(food_long, new_food, how = "left", on="Date").drop(columns=["Date"])

In [15]:
fact_food.to_sql(name="fact_food", con=engine, if_exists='append', index=False)

1403

# Ad-Hoc

## Inputting new 10RM workout plan

Rethink how I'm loading in new 10rm workouits to put through this workflow

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

with engine.connect() as connection:
    workouts_raw = pd.read_sql_query(
        text("""SELECT ha.*, he.log_date
                FROM habit_answers ha
                left join habit_entries he
                on he.id = ha.entry_id
                left join habits h
                on h.id = he.habit_id
                WHERE h.name = 'workout'"""), connection)

# Making the dataset long for easy filtering, can rescuce the index columns but just reseting the index and now they magically appear as regular columns again!
workouts = pd.pivot(workouts_raw, index=['entry_id', 'log_date'], columns='question', values='answer').reset_index()

workouts_filt = workouts[workouts["10RM"] == "True"]

# Now that I have only 10RM date I can get to my 10rm data however is necessary

rm_10_workouts = workouts_filt[workouts_filt ['log_date'].isin(["2026-03-07", "2026-03-08"]) ]

In [ ]:
columns = ['entry_id', '10RM', 'comment', 'effort', 'reps', 'sets', 'weight', 'workout_type']
workout_df_total = pd.DataFrame(columns=columns)

entry_ids = rm_10_workouts['entry_id'].unique().tolist()

with engine.connect() as connection:
    for id in entry_ids:
        workout_df = pd.read_sql_query(text("SELECT * FROM habit_answers ha WHERE entry_id = :id"), connection, params={"id": id})

        workout_df = workout_df[['entry_id', 'question', 'answer']]

        workout_df = workout_df.pivot(index='entry_id', columns='question', values='answer').reset_index()

        workout_df_total = pd.concat([workout_df_total, workout_df], ignore_index=True)

    habit_entries = pd.read_sql_query(text("SELECT * FROM habit_entries"), connection)

    workout_df_total = workout_df_total.merge(habit_entries, left_on='entry_id', right_on='id', how='left')

    workout_df_total = workout_df_total[["log_date", "workout_type", "weight", "sets", "reps", "effort", "comment"]]

In [ ]:
# Use the new workouts df to calculate 10RMs
import pytz
from datetime import datetime

# Now that I have a list of 10RM workouts, I want to multiply them by the specific percentage over each week 
multipliers = {"Week 1": 0.25,
    "Week 2": 0.30,
    "Week 3": 0.35,
    "Week 4": 0.40,
    "Week 5": 0.60,
    "Week 6": 0.65,
    "Week 7": 0.70,
    "Week 8": 0.75,
    "Week 9": 0.85,
    "Week 10": 0.90,
    "Week 11": 0.95,
    "Week 12": 1}

eastern = pytz.timezone("America/New_York")
date = datetime.now(eastern).strftime("%Y-%m-%d")

workout_df_total = rm_10_workouts.copy()

workout_df_total['weight'] = pd.to_numeric(workout_df_total['weight'])

rm10_plan = pd.DataFrame(columns=["week_number", "exercise_name", "target_weight", "created_date"])

for _, df_row in workout_df_total.iterrows():
    activity_name = df_row["workout_type"]
    activity_weight = df_row["weight"]

    for i, row_dict in enumerate(multipliers.items()):
        week = row_dict[0] 
        multiplier = row_dict[1]
        
        new_weight = activity_weight * multiplier

        #Temporary dictionay to store the data 
        tempdict = {
        "week_number": week,
        "exercise_name": activity_name,
        "target_weight": new_weight,
        "created_date": date} 

        tempdf = pd.DataFrame(tempdict, index=[0])

        rm10_plan = pd.concat([rm10_plan, tempdf], ignore_index=True)

In [ ]:
# Apply reps and sets based on the week number
import numpy as np

def assign_reps_sets(week):
    if week in ["Week 1", "Week 2", "Week 3", "Week 4"]:
        return 3, "30/25/20"
    elif week in ["Week 5", "Week 6", "Week 7", "Week 8"]:
        return 3, 10
    elif week in ["Week 9", "Week 10", "Week 11", "Week 12"]:
        return 3, "5/3/1"
    else:
        return None, None

rm10_plan[['sets', 'reps']] = rm10_plan['week_number'].apply(lambda x: pd.Series(assign_reps_sets(x)))

In [ ]:
# applying workout type to newly created 10RM plan
import numpy as np

rm10_plan["workout_type"] = np.where(rm10_plan["exercise_name"].str.contains("Chest|Triceps", regex = True), "Push", 
                                    np.where(rm10_plan["exercise_name"].str.contains("Back|Biceps|Shoulders", regex = True), "Pull", 
                                             np.where(rm10_plan["exercise_name"].str.contains("Hip|Leg|Calf", regex = True), "Legs", "unknown")))

# Strip string from week number
rm10_plan["week_number"] = rm10_plan["week_number"].str.replace("Week ", "").astype(int)

# including some logic to round to nearest 5 lbs because that's what the machines use 
rm10_plan["target_weight_rounded"]= 5 * (rm10_plan["target_weight"] / 5).round()

In [ ]:
# Upload plan data to sql db
engine = create_engine(f"sqlite:///habits.db")
for _, row in rm10_plan.iterrows():

    with engine.begin() as conn:
        conn.execute(text("""INSERT INTO tenrm_plans (week_number, exercise_name, target_weight, created_date , sets, reps, workout_type)
        VALUES (:week_number, :exercise_name, :target_weight, :created_date , :sets, :reps, :workout_type)"""),
        {"week_number": row["week_number"], "exercise_name": row["exercise_name"], "target_weight": row["target_weight_rounded"], "created_date": row["created_date"], "sets": row["sets"], "reps": row["reps"], "workout_type": row["workout_type"]})

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(r"sqlite:///\\192.168.1.164\WarpDrive\habit_website\habits.db")

with engine.begin() as conn:
    hello = pd.read_sql("""SELECT * FROM tenrm_plans""", conn)

# Graveyard

## Timezone migration code

### Converting apple raw db to UTC format

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    apple_data_raw = pd.read_sql_query(
        text("""SELECT * FROM apple_data_raw"""),
        connection)

# (30 seconds)
datetime_cols = ['startDate', 'endDate', 'creationDate']
for col in datetime_cols:
    apple_data_raw[col] = pd.to_datetime(apple_data_raw[col], format="%Y-%m-%dT%H:%M:%S%z", utc=True)

# Appending the new raw data to my sql lite table (2 min 12 secs)
apple_data_raw.to_sql(name="apple_data_raw", con=engine, if_exists='replace', index=False)

### Converting apple workouts db to UTC format

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    apple_workouts = pd.read_sql_query(
        text("""SELECT * FROM apple_workouts"""), 
        connection)


datetime_cols = ['StartDate']
for col in datetime_cols:
    apple_workouts[col] = pd.to_datetime(apple_workouts[col], format="%Y-%m-%d %H:%M:%S%z", utc=True)


# Appending the new raw data to my sql lite table
apple_workouts.to_sql(name="apple_workouts", con=engine, if_exists='replace', index=False)

### Fix timezones in habit entries

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    habit_entries = pd.read_sql_query(
        text("""SELECT * FROM habit_entries"""), connection)
        #parse_dates=["timestamp"]) # Verifying that conversion works

# Converting timestamp from EST to UTC
#habit_entries["timestamp"] = pd.to_datetime(habit_entries["timestamp"]).dt.tz_localize("US/Eastern").dt.tz_convert("UTC")
habit_entries["timestamp"] = pd.to_datetime(habit_entries["timestamp"], format= "%Y-%m-%d %H:%M:%S", utc = True)

# Appending the new raw data to my sql lite table
habit_entries.to_sql(name="habit_entries", con=engine, if_exists='replace', index=False)

# Need to now recreate the habit entries table with correct keys, nulls, and autoincrements
with engine.connect() as conn:
    # habit_entries table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS habit_entries_new (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        habit_id INTEGER NOT NULL,
                        log_date TEXT NOT NULL,
                        timestamp TEXT NOT NULL,
                        FOREIGN KEY (habit_id) REFERENCES habits(id));"""))

In [ ]:
with engine.begin() as conn:
    # habit_entries table
    conn.execute(text("""INSERT INTO habit_entries_new (id, habit_id, log_date, timestamp)
                        SELECT id, habit_id, log_date, timestamp
                        FROM habit_entries"""))
    
# Delete original habit_entries table from sqlite and rename "habit_entries_new" to "habit_entries" 

### Fix timezones in 10RM completions

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    tenrm_completions = pd.read_sql_query(
        text("""SELECT * FROM tenrm_completions"""), connection)
        #parse_dates=["timestamp"]) # Verifying that conversion works

# Converting timestamp from EST to UTC
#tenrm_completions["timestamp"] = pd.to_datetime(tenrm_completions["timestamp"]).dt.tz_localize("US/Eastern").dt.tz_convert("UTC")
tenrm_completions["timestamp"] = pd.to_datetime(tenrm_completions["timestamp"], format= "%Y-%m-%d %H:%M:%S", utc = True)

# Appending the new raw data to my sql lite table
tenrm_completions.to_sql(name="tenrm_completions", con=engine, if_exists='replace', index=False)

In [ ]:
# Need to now recreate the tenrm_completions table with correct keys, nulls, and autoincrements
with engine.connect() as conn:
    # habit_entries table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS tenrm_completions_new (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        plan_id INTEGER NOT NULL,
                        completion_date TEXT NOT NULL,
                        completed INTEGER NOT NULL,
                        notes TEXT,
                        timestamp TEXT NOT NULL,
                        FOREIGN KEY (plan_id) REFERENCES tenrm_plans(id));"""))

In [ ]:
with engine.begin() as conn:
    # tenrm_completions table
    conn.execute(text("""INSERT INTO tenrm_completions_new (id, plan_id, completion_date, completed, notes, timestamp)
                        SELECT id, plan_id, completion_date, completed, notes, timestamp
                        FROM tenrm_completions"""))
    
# Delete original tenrm_completions table from sqlite and rename "tenrm_completions_new" to "tenrm_completions" 

### Fix timezones in access logs table

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo

engine = create_engine(f"sqlite:///habits.db")

# Read from SQLite and parse those columns as datetime
with engine.connect() as connection:
    access_log = pd.read_sql_query(
        text("""SELECT * FROM access_log"""), connection)
        #parse_dates=["timestamp"]) # Verifying that conversion works

# Converting timestamp from EST to UTC
#tenrm_completions["timestamp"] = pd.to_datetime(tenrm_completions["timestamp"]).dt.tz_localize("US/Eastern").dt.tz_convert("UTC")
access_log["timestamp"] = pd.to_datetime(access_log["timestamp"], format= "%Y-%m-%d %H:%M:%S", utc = True)

# Appending the new raw data to my sql lite table
access_log.to_sql(name="access_log", con=engine, if_exists='replace', index=False)

In [ ]:
# Need to now recreate the tenrm_completions table with correct keys, nulls, and autoincrements
with engine.connect() as conn:
    # access_log table
    conn.execute(text("""CREATE TABLE IF NOT EXISTS access_log_new (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        ip TEXT NOT NULL,
                        endpoint TEXT NOT NULL,
                        timestamp TEXT NOT NULL);"""))

In [ ]:
with engine.begin() as conn:
    # tenrm_completions table
    conn.execute(text("""INSERT INTO access_log_new (id, ip, endpoint, timestamp)
                        SELECT id, ip, endpoint, timestamp
                        FROM access_log"""))
    
# Delete original access_log table from sqlite and rename "access_log_new" to "access_log" 